In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. LOAD DATA
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

target = "pikachu_hp"
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

# Track original index ordering for test predictions
test["orig_id"] = np.arange(len(test))


# 2. FEATURE ENGINEERING & LAG PIPELINE
def process_pipeline(df):
    df = df.copy()

    # Categorical Cleaning
    cat_cols = df.select_dtypes(include=["object"]).columns
    for col in cat_cols:
        df[col] = df[col].fillna("Unknown").astype(str).str.strip()

    # Basic HP / Damage
    df["damage_pct"] = df["damage_dealt"] / df["max_hp"].replace(0, np.nan)
    df["healing_pct"] = df["healing_applied"] / df["max_hp"].replace(0, np.nan)
    df["previous_hp_pct"] = df["previous_hp"] / df["max_hp"].replace(0, np.nan)

    # Type Matchup & Effectiveness
    df["move_vs_opponent"] = (
        df["move_type"].fillna("Unknown")
        + "_vs_"
        + df["opponent_type"].fillna("Unknown")
    )
    df["effective_move_power"] = df["move_power"] * df["type_effectiveness"]
    df["effective_damage_potential"] = (
        df["move_power"] * df["type_effectiveness"] * df["move_hit"]
    )

    # Stats Differences & Ratios
    df["level_difference"] = df["pikachu_level"] - df["opponent_level"]
    df["level_ratio"] = df["pikachu_level"] / df["opponent_level"].replace(
        0, np.nan
    )
    df["attack_difference"] = df["attack_stat"] - df["defense_stat"]
    df["attack_ratio"] = df["attack_stat"] / df["defense_stat"].replace(
        0, np.nan
    )
    df["sp_attack_difference"] = df["sp_attack_stat"] - df["sp_defense_stat"]
    df["sp_attack_ratio"] = (
        df["sp_attack_stat"] / df["sp_defense_stat"].replace(0, np.nan)
    )
    df["speed_difference"] = (
        df["speed_stat_pikachu"] - df["speed_stat_opponent"]
    )
    df["speed_ratio"] = df["speed_stat_pikachu"] / df[
        "speed_stat_opponent"
    ].replace(0, np.nan)

    # Stage Effects
    df["attack_stage_effect"] = df["attack_stat"] * df["attack_stage"]
    df["defense_stage_effect"] = df["defense_stat"] * df["defense_stage"]
    df["speed_stage_effect"] = df["speed_stat_pikachu"] * df["speed_stage"]
    df["effective_attack_vs_defense"] = (
        df["attack_stat"] * df["attack_stage"]
    ) / (df["defense_stat"] * df["defense_stage"]).replace(0, np.nan)

    # Interactions
    df["move_power_x_attack"] = df["move_power"] * df["attack_stat"]
    df["move_power_x_level"] = df["move_power"] * df["pikachu_level"]
    df["effectiveness_x_power"] = df["type_effectiveness"] * df["move_power"]
    df["effectiveness_x_attack"] = df["type_effectiveness"] * df["attack_stat"]
    df["effectiveness_x_level"] = df["type_effectiveness"] * df["pikachu_level"]

    # Time & Weather / Terrain Categorical Interactions
    df["turn_squared"] = df["turn"] ** 2
    df["turn_sqrt"] = np.sqrt(df["turn"].clip(lower=0))
    df["move_category_weather"] = (
        df["move_category"].fillna("Unknown")
        + "_"
        + df["weather_condition"].fillna("Unknown")
    )
    df["move_category_terrain"] = (
        df["move_category"].fillna("Unknown")
        + "_"
        + df["terrain_type"].fillna("Unknown")
    )
    df["status_weather"] = (
        df["pikachu_status"].fillna("Unknown")
        + "_"
        + df["weather_condition"].fillna("Unknown")
    )
    df["ability_weather"] = (
        df["pikachu_ability"].fillna("Unknown")
        + "_"
        + df["weather_condition"].fillna("Unknown")
    )
    df["ability_terrain"] = (
        df["pikachu_ability"].fillna("Unknown")
        + "_"
        + df["terrain_type"].fillna("Unknown")
    )

    # Temporal Sort before generating Lags
    df = df.sort_values(["round", "turn"])

    # 1-Turn Lag Features
    lag_cols = [
        "damage_dealt",
        "healing_applied",
        "previous_hp",
        "type_effectiveness",
        "move_power",
    ]
    for col in lag_cols:
        df[f"{col}_lag1"] = df.groupby("round")[col].shift(1)

    # 2-Turn Lag Features
    lag2_cols = ["damage_dealt", "healing_applied", "previous_hp"]
    for col in lag2_cols:
        df[f"{col}_lag2"] = df.groupby("round")[col].shift(2)

    # Rolling Damage
    df["damage_last_3"] = df.groupby("round")["damage_dealt"].transform(
        lambda x: x.shift(1).rolling(3).sum()
    )

    # Restore Original Order if test set, or reset index if train set
    if "orig_id" in df.columns:
        df = df.sort_values("orig_id").drop(columns=["orig_id"])
    else:
        df = df.reset_index(drop=True)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df


# 3. RUN PIPELINE
train = process_pipeline(train)
test = process_pipeline(test)

X = train.drop(columns=[target, "battle_turn"], errors="ignore")
y = train[target]
X_test = test.drop(columns=["battle_turn"], errors="ignore")

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# 4. PREPROCESSING PIPELINE
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# 5. MODEL SELECTION & PREDICTION
model = Pipeline(
    [("preprocessor", preprocessor), ("regression", LinearRegression())]
)

model.fit(X, y)
test_preds = model.predict(X_test)

# Clip HP to minimum 0
test_preds = np.clip(test_preds, a_min=0, a_max=None)

submission = pd.DataFrame(
    {"battle_turn": sample["battle_turn"], "pikachu_hp": test_preds}
)

submission.to_csv("submission_5.csv", index=False)

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_28792\496019877.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns
C:\Users\Bhavin\AppData\Local\Temp\ipykernel_28792\496019877.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gui

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_focus = train[["trainer_focus_score"]]
y = train["pikachu_hp"]

model_focus = LinearRegression()

model_focus.fit(X_focus, y)

train_pred = model_focus.predict(X_focus)

print("Training R²:",
      r2_score(y, train_pred))

print("\nCoefficient:",
      model_focus.coef_[0])

print("Intercept:",
      model_focus.intercept_)

Training R²: 0.9254399887660985

Coefficient: 0.7925108181407473
Intercept: -10.818234973057791


In [4]:
test_focus_pred = model_focus.predict(
    test[["trainer_focus_score"]]
)

print("\nTest predictions:")
print(
    pd.Series(test_focus_pred).describe()
)


Test predictions:
count    240.000000
mean      33.324287
std       18.222848
min        1.386432
25%       18.821670
50%       33.364243
75%       48.719140
max       64.470293
dtype: float64


In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OrdinalEncoder

# 1. LOAD DATA
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

target = "pikachu_hp"
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

# Track original index ordering for test predictions
test["orig_id"] = np.arange(len(test))

# 2. FEATURE ENGINEERING & LAG PIPELINE
def process_pipeline(df):
    df = df.copy()

    # Temporal Sort before generating Lags ensures we don't peek at the future
    df = df.sort_values(["round", "turn"]).reset_index(drop=True)

    # Basic HP / Damage
    df["damage_pct"] = df["damage_dealt"] / df["max_hp"].replace(0, np.nan)
    df["healing_pct"] = df["healing_applied"] / df["max_hp"].replace(0, np.nan)
    
    # Type Matchup & Effectiveness
    df["effective_move_power"] = df["move_power"] * df["type_effectiveness"]
    df["effective_damage_potential"] = (
        df["move_power"] * df["type_effectiveness"] * df["move_hit"]
    )

    # Stats Differences (Trees handle basic differences much better than complex ratios)
    df["level_difference"] = df["pikachu_level"] - df["opponent_level"]
    df["attack_difference"] = df["attack_stat"] - df["defense_stat"]
    df["speed_difference"] = df["speed_stat_pikachu"] - df["speed_stat_opponent"]

    # 1-Turn & 2-Turn Lag Features (Grouped by battle round)
    lag_cols = ["damage_dealt", "healing_applied", "previous_hp", "type_effectiveness", "move_power"]
    for col in lag_cols:
        df[f"{col}_lag1"] = df.groupby("round")[col].shift(1)
        df[f"{col}_lag2"] = df.groupby("round")[col].shift(2)

    # HP Momentum & Cumulative Features
    df["hp_change_lag1"] = df["previous_hp"] - df["previous_hp_lag1"]
    df["cum_damage"] = df.groupby("round")["damage_dealt"].cumsum()
    df["cum_healing"] = df.groupby("round")["healing_applied"].cumsum()
    
    # The Absolute Anchor Feature
    # Pikachu's current HP is heavily dependent on the basic math of the previous turn.
    df["math_expected_hp"] = df["previous_hp"] - df["damage_dealt"] + df["healing_applied"]

    # Restore Original Order if test set, or reset index if train set
    if "orig_id" in df.columns:
        df = df.sort_values("orig_id").drop(columns=["orig_id"])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

# 3. RUN PIPELINE
train_processed = process_pipeline(train)
test_processed = process_pipeline(test)

# 4. CATEGORICAL ENCODING
# We use OrdinalEncoder instead of OneHotEncoder to prevent high cardinality issues
cat_cols = train_processed.select_dtypes(include=["object"]).columns.tolist()
if "battle_turn" in cat_cols: cat_cols.remove("battle_turn")

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_processed[cat_cols] = encoder.fit_transform(train_processed[cat_cols].fillna("Unknown"))
test_processed[cat_cols] = encoder.transform(test_processed[cat_cols].fillna("Unknown"))

# 5. MODEL SETUP & CV
features = [c for c in train_processed.columns if c not in [target, "battle_turn"]]
categorical_features_idx = [features.index(c) for c in cat_cols]

X = train_processed[features]
y = train_processed[target]
groups = train_processed["round"]

# GroupKFold prevents leakage across different battles
gkf = GroupKFold(n_splits=5)
test_preds = np.zeros(len(test_processed))

model = HistGradientBoostingRegressor(
    categorical_features=categorical_features_idx,
    learning_rate=0.05,
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

# 6. TRAIN & PREDICT
print("Training K-Fold ensemble...")
for fold, (trn_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, y_train = X.iloc[trn_idx], y.iloc[trn_idx]
    
    model.fit(X_train, y_train)
    # Accumulate predictions on the test set
    test_preds += model.predict(test_processed[features]) / gkf.n_splits

# Clip HP to minimum 0 (Pikachu can't have negative HP)
test_preds = np.clip(test_preds, a_min=0, a_max=None)

submission = pd.DataFrame(
    {"battle_turn": sample["battle_turn"], "pikachu_hp": test_preds}
)

submission.to_csv("submissionDay6.csv", index=False)
print("Submission saved successfully!")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_7360\3757137840.py:69: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_processed.select_dtypes(include=["object"]).columns.tolist()


Training K-Fold ensemble...
Submission saved successfully!
